# Vietnamese Restaurant Review Sentiment - Final Overall ML

Final Kaggle notebook focused on clean overall metrics for a 3-class restaurant review sentiment task.

Approach: classical Machine Learning only, using TF-IDF word/character features and linear classifiers. The model is selected by a balanced overall score based on Accuracy, Macro F1 and Weighted F1.

In [ ]:
from pathlib import Path
import json
import re

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression, RidgeClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
    precision_recall_fscore_support,
)
from sklearn.pipeline import FeatureUnion, Pipeline
from sklearn.svm import LinearSVC

RANDOM_STATE = 42
OUTPUT_DIR = Path('/kaggle/working')
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_ROOT_CANDIDATES = [
    Path('/kaggle/input/datasets/thuanminh1310/restaurant/vietnamese-restaurant-review-sentiment-dataset'),
    Path('/kaggle/input/restaurant/vietnamese-restaurant-review-sentiment-dataset'),
    Path('/kaggle/input/vietnamese-restaurant-review-sentiment-dataset'),
]

LABEL_MAP = {0: 'negative', 1: 'positive', 2: 'neutral'}
SENTIMENT_LABELS = ['negative', 'neutral', 'positive']
LABEL_COLORS = {'negative': '#d62728', 'neutral': '#ffbf00', 'positive': '#2ca02c'}

FAST_DEBUG = False
MAX_WORD_FEATURES = 160_000
MAX_CHAR_FEATURES = 100_000

# Overall-first selection. This prefers strong total metrics over aggressive neutral recall.
OVERALL_SCORE_WEIGHTS = {
    'macro_f1': 0.40,
    'weighted_f1': 0.35,
    'accuracy': 0.25,
}
NEUTRAL_CLASS_WEIGHT_SCALES = [1.0, 1.1, 1.25]
NEUTRAL_OVERSAMPLE_FACTORS = [1.0, 1.15]

sns.set_theme(style='whitegrid', font_scale=1.05)
print('Ready')

In [ ]:
def find_data_root(candidates):
    for root in candidates:
        data_dir = root / 'data'
        if data_dir.exists() and list(data_dir.glob('train-*.parquet')):
            return root
    print('Available files under /kaggle/input:')
    input_root = Path('/kaggle/input')
    if input_root.exists():
        for p in sorted(input_root.rglob('*')):
            if p.is_file():
                print('-', p)
    raise FileNotFoundError('Dataset root not found. Check DATA_ROOT_CANDIDATES.')


def load_split(data_root, split):
    matches = sorted((data_root / 'data').glob(f'{split}-*.parquet'))
    if not matches:
        raise FileNotFoundError('Missing split file: ' + split)
    df = pd.read_parquet(matches[0]).copy()
    required = {'review_id', 'review', 'label'}
    missing = required - set(df.columns)
    if missing:
        raise ValueError('Missing columns: ' + str(sorted(missing)))
    df['split'] = split
    return df


def normalize_text(value):
    text = '' if pd.isna(value) else str(value)
    text = text.lower()
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text


def prepare_split(df):
    out = df.copy()
    out['review_clean'] = out['review'].map(normalize_text)
    out['sentiment_label'] = out['label'].map(LABEL_MAP)
    if out['sentiment_label'].isna().any():
        bad = sorted(out.loc[out['sentiment_label'].isna(), 'label'].dropna().unique().tolist())
        raise ValueError('Unknown label values: ' + str(bad))
    out = out[out['review_clean'].str.len() > 0].copy()
    return out


data_root = find_data_root(DATA_ROOT_CANDIDATES)
print('Data root:', data_root)

train_df = prepare_split(load_split(data_root, 'train'))
val_df = prepare_split(load_split(data_root, 'validation'))
test_df = prepare_split(load_split(data_root, 'test'))

print('Raw shapes:', train_df.shape, val_df.shape, test_df.shape)
display(train_df.head())

In [ ]:
def clean_duplicates_and_leakage(train_df, val_df, test_df):
    stats = {
        'train_before': len(train_df),
        'validation_before': len(val_df),
        'test_before': len(test_df),
    }

    all_df = pd.concat([train_df, val_df, test_df], ignore_index=True)
    conflict_texts = set(
        all_df.groupby('review_clean')['sentiment_label']
        .nunique()
        .loc[lambda s: s > 1]
        .index
    )

    train_df = train_df[~train_df['review_clean'].isin(conflict_texts)].copy()
    val_df = val_df[~val_df['review_clean'].isin(conflict_texts)].copy()
    test_df = test_df[~test_df['review_clean'].isin(conflict_texts)].copy()

    train_df = train_df.drop_duplicates('review_clean', keep='first').copy()
    val_df = val_df.drop_duplicates('review_clean', keep='first').copy()
    test_df = test_df.drop_duplicates('review_clean', keep='first').copy()

    train_texts = set(train_df['review_clean'])
    val_df = val_df[~val_df['review_clean'].isin(train_texts)].copy()

    train_val_texts = train_texts | set(val_df['review_clean'])
    test_df = test_df[~test_df['review_clean'].isin(train_val_texts)].copy()

    stats.update({
        'conflicting_review_texts_removed': len(conflict_texts),
        'train_after': len(train_df),
        'validation_after': len(val_df),
        'test_after': len(test_df),
    })
    return train_df.reset_index(drop=True), val_df.reset_index(drop=True), test_df.reset_index(drop=True), stats


def fast_debug_subset(df, n_per_class):
    return (
        df.groupby('sentiment_label', group_keys=False)
        .apply(lambda part: part.sample(min(len(part), n_per_class), random_state=RANDOM_STATE))
        .sample(frac=1.0, random_state=RANDOM_STATE)
        .reset_index(drop=True)
    )


train_df, val_df, test_df, cleaning_stats = clean_duplicates_and_leakage(train_df, val_df, test_df)

if FAST_DEBUG:
    train_df = fast_debug_subset(train_df, 400)
    val_df = fast_debug_subset(val_df, 120)
    test_df = fast_debug_subset(test_df, 120)

print('Cleaning stats:')
print(json.dumps(cleaning_stats, indent=2, ensure_ascii=False))

summary_df = pd.concat([
    train_df['sentiment_label'].value_counts().rename('train'),
    val_df['sentiment_label'].value_counts().rename('validation'),
    test_df['sentiment_label'].value_counts().rename('test'),
], axis=1).reindex(SENTIMENT_LABELS)

display(summary_df)
display((summary_df / summary_df.sum(axis=0) * 100).round(2))

summary_plot_df = (
    summary_df.rename_axis('sentiment_label').reset_index()
    .melt(id_vars='sentiment_label', var_name='split', value_name='count')
)
plt.figure(figsize=(9, 5))
sns.barplot(
    data=summary_plot_df,
    x='split',
    y='count',
    hue='sentiment_label',
    hue_order=SENTIMENT_LABELS,
    palette=[LABEL_COLORS[label] for label in SENTIMENT_LABELS],
)
plt.title('Class distribution by split after cleaning')
plt.xlabel('Split')
plt.ylabel('Number of reviews')
plt.legend(title='Sentiment')
plt.tight_layout()
class_distribution_path = OUTPUT_DIR / 'final_overall_class_distribution.png'
plt.savefig(class_distribution_path, dpi=160, bbox_inches='tight')
plt.show()

In [ ]:
features = FeatureUnion([
    ('word_tfidf', TfidfVectorizer(
        analyzer='word',
        ngram_range=(1, 3),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        max_features=MAX_WORD_FEATURES,
    )),
    ('char_tfidf', TfidfVectorizer(
        analyzer='char_wb',
        ngram_range=(3, 5),
        min_df=2,
        max_df=0.95,
        sublinear_tf=True,
        max_features=MAX_CHAR_FEATURES,
    )),
])

X_train_text = train_df['review_clean']
y_train = train_df['sentiment_label']
X_val_text = val_df['review_clean']
y_val = val_df['sentiment_label']
X_test_text = test_df['review_clean']
y_test = test_df['sentiment_label']

print('Fitting TF-IDF features once for validation search...')
X_train_vec = features.fit_transform(X_train_text)
X_val_vec = features.transform(X_val_text)
X_test_vec = features.transform(X_test_text)

print('X_train_vec:', X_train_vec.shape)
print('X_val_vec  :', X_val_vec.shape)
print('X_test_vec :', X_test_vec.shape)

In [ ]:
def evaluate_predictions(y_true, y_pred, split):
    precision, recall, f1, support = precision_recall_fscore_support(
        y_true,
        y_pred,
        labels=SENTIMENT_LABELS,
        zero_division=0,
    )
    row = {
        'split': split,
        'accuracy': accuracy_score(y_true, y_pred),
        'macro_f1': f1_score(y_true, y_pred, average='macro', labels=SENTIMENT_LABELS),
        'weighted_f1': f1_score(y_true, y_pred, average='weighted', labels=SENTIMENT_LABELS),
    }
    for label, p, r, f, s in zip(SENTIMENT_LABELS, precision, recall, f1, support):
        row[f'{label}_precision'] = p
        row[f'{label}_recall'] = r
        row[f'{label}_f1'] = f
        row[f'{label}_support'] = int(s)
    return row


def overall_score(row):
    return (
        OVERALL_SCORE_WEIGHTS['macro_f1'] * row['macro_f1']
        + OVERALL_SCORE_WEIGHTS['weighted_f1'] * row['weighted_f1']
        + OVERALL_SCORE_WEIGHTS['accuracy'] * row['accuracy']
    )


def make_balanced_class_weight(y, neutral_scale=1.0):
    counts = pd.Series(y).value_counts()
    total = len(y)
    n_classes = len(SENTIMENT_LABELS)
    weights = {label: total / (n_classes * counts[label]) for label in SENTIMENT_LABELS}
    weights['neutral'] *= neutral_scale
    return weights


def make_classifier(classifier_name, class_weight):
    if classifier_name == 'ridge_alpha1':
        return RidgeClassifier(alpha=1.0, class_weight=class_weight)
    if classifier_name == 'ridge_alpha05':
        return RidgeClassifier(alpha=0.5, class_weight=class_weight)
    if classifier_name == 'linear_svc_c1':
        return LinearSVC(C=1.0, class_weight=class_weight, max_iter=10_000, random_state=RANDOM_STATE)
    if classifier_name == 'linear_svc_c05':
        return LinearSVC(C=0.5, class_weight=class_weight, max_iter=10_000, random_state=RANDOM_STATE)
    if classifier_name == 'logreg_c2':
        return LogisticRegression(
            C=2.0,
            class_weight=class_weight,
            max_iter=2_000,
            random_state=RANDOM_STATE,
            solver='liblinear',
        )
    raise ValueError('Unknown classifier: ' + classifier_name)


def oversampled_indices(y, neutral_factor):
    y_array = np.asarray(y)
    base_idx = np.arange(len(y_array))
    if neutral_factor <= 1.0:
        return base_idx
    rng = np.random.default_rng(RANDOM_STATE + int(neutral_factor * 100))
    neutral_idx = np.flatnonzero(y_array == 'neutral')
    extra_count = int(round(len(neutral_idx) * (neutral_factor - 1.0)))
    extra_idx = rng.choice(neutral_idx, size=extra_count, replace=True)
    return rng.permutation(np.concatenate([base_idx, extra_idx]))


classifier_names = ['ridge_alpha1', 'ridge_alpha05', 'linear_svc_c1', 'linear_svc_c05', 'logreg_c2']
validation_rows = []
fitted_classifiers = {}
config_specs = {}
y_train_array = y_train.to_numpy()

for neutral_scale in NEUTRAL_CLASS_WEIGHT_SCALES:
    class_weight = make_balanced_class_weight(y_train_array, neutral_scale=neutral_scale)
    weight_name = f"neutral_x{str(neutral_scale).replace('.', 'p')}"

    for neutral_factor in NEUTRAL_OVERSAMPLE_FACTORS:
        train_idx = oversampled_indices(y_train_array, neutral_factor)
        os_name = f"os{str(neutral_factor).replace('.', 'p')}"

        for classifier_name in classifier_names:
            config_name = f'{classifier_name}__{weight_name}__{os_name}'
            print('Training classifier:', config_name)
            clf = make_classifier(classifier_name, class_weight)
            clf.fit(X_train_vec[train_idx], y_train_array[train_idx])
            pred_val = clf.predict(X_val_vec)

            row = evaluate_predictions(y_val, pred_val, 'validation')
            row['config'] = config_name
            row['classifier'] = classifier_name
            row['neutral_weight_scale'] = neutral_scale
            row['neutral_oversample_factor'] = neutral_factor
            row['selection_score'] = overall_score(row)
            validation_rows.append(row)
            fitted_classifiers[config_name] = clf
            config_specs[config_name] = {
                'classifier_name': classifier_name,
                'neutral_weight_scale': neutral_scale,
                'neutral_oversample_factor': neutral_factor,
                'class_weight': class_weight,
            }
            print(
                f"  score={row['selection_score']:.4f} accuracy={row['accuracy']:.4f} "
                f"macro_f1={row['macro_f1']:.4f} weighted_f1={row['weighted_f1']:.4f} "
                f"neutral_f1={row['neutral_f1']:.4f}"
            )

validation_results_df = pd.DataFrame(validation_rows).sort_values(
    ['selection_score', 'accuracy', 'weighted_f1', 'macro_f1'],
    ascending=[False, False, False, False],
)
best_config = validation_results_df.iloc[0]['config']
best_clf = fitted_classifiers[best_config]
best_config_details = config_specs[best_config]

print('Best config:', best_config)
display(validation_results_df[[
    'config',
    'selection_score',
    'accuracy',
    'macro_f1',
    'weighted_f1',
    'negative_f1',
    'neutral_f1',
    'positive_f1',
    'neutral_weight_scale',
    'neutral_oversample_factor',
]].head(20))

In [ ]:
train_pred = best_clf.predict(X_train_vec)
val_pred = best_clf.predict(X_val_vec)

train_metrics = evaluate_predictions(y_train, train_pred, 'train')
val_metrics = evaluate_predictions(y_val, val_pred, 'validation')

train_plus_val_df = pd.concat([train_df, val_df], ignore_index=True)
X_train_val_text = train_plus_val_df['review_clean']
y_train_val = train_plus_val_df['sentiment_label']
y_train_val_array = y_train_val.to_numpy()

print('Refitting TF-IDF on train + validation for final model...')
final_features = clone(features)
X_train_val_vec = final_features.fit_transform(X_train_val_text)
X_test_final_vec = final_features.transform(X_test_text)

final_class_weight = make_balanced_class_weight(
    y_train_val_array,
    neutral_scale=best_config_details['neutral_weight_scale'],
)
final_train_idx = oversampled_indices(
    y_train_val_array,
    best_config_details['neutral_oversample_factor'],
)
final_clf = make_classifier(best_config_details['classifier_name'], final_class_weight)
final_clf.fit(X_train_val_vec[final_train_idx], y_train_val_array[final_train_idx])
final_test_pred = final_clf.predict(X_test_final_vec)
final_test_metrics = evaluate_predictions(y_test, final_test_pred, 'final_test')

metrics_df = pd.DataFrame([train_metrics, val_metrics, final_test_metrics])
display(metrics_df[['split', 'accuracy', 'macro_f1', 'weighted_f1', 'negative_f1', 'neutral_f1', 'positive_f1']])

print('Final test classification report:')
print(classification_report(y_test, final_test_pred, labels=SENTIMENT_LABELS, digits=4, zero_division=0))

report_df = pd.DataFrame(
    classification_report(
        y_test,
        final_test_pred,
        labels=SENTIMENT_LABELS,
        output_dict=True,
        zero_division=0,
    )
).T
cm = confusion_matrix(y_test, final_test_pred, labels=SENTIMENT_LABELS)
cm_df = pd.DataFrame(cm, index=SENTIMENT_LABELS, columns=SENTIMENT_LABELS)
cm_norm_df = cm_df.div(cm_df.sum(axis=1), axis=0).fillna(0)
display(cm_df)

prediction_df = test_df[['review_id', 'review', 'review_clean', 'sentiment_label']].copy()
prediction_df['pred_sentiment_label'] = final_test_pred
prediction_df['is_correct'] = prediction_df['sentiment_label'] == prediction_df['pred_sentiment_label']

In [ ]:
top_validation_plot_df = validation_results_df.head(15).copy()
top_validation_plot_df['rank'] = [f'#{i}' for i in range(1, len(top_validation_plot_df) + 1)]
validation_metric_plot_df = top_validation_plot_df.melt(
    id_vars=['rank'],
    value_vars=['selection_score', 'accuracy', 'weighted_f1', 'macro_f1'],
    var_name='metric',
    value_name='score',
)
plt.figure(figsize=(11, 6))
sns.barplot(data=validation_metric_plot_df, y='rank', x='score', hue='metric')
plt.title('Top validation configurations - overall selection')
plt.xlabel('Score')
plt.ylabel('Validation rank')
plt.xlim(0, 1)
plt.legend(title='Metric', loc='lower right')
plt.tight_layout()
validation_comparison_path = OUTPUT_DIR / 'final_overall_validation_model_comparison.png'
plt.savefig(validation_comparison_path, dpi=160, bbox_inches='tight')
plt.show()

class_metrics_df = report_df.loc[SENTIMENT_LABELS, ['precision', 'recall', 'f1-score']].copy()
class_metrics_df = class_metrics_df.rename_axis('sentiment_label').reset_index()
class_metrics_plot_df = class_metrics_df.melt(
    id_vars='sentiment_label',
    var_name='metric',
    value_name='score',
)
plt.figure(figsize=(9, 5))
sns.barplot(
    data=class_metrics_plot_df,
    x='sentiment_label',
    y='score',
    hue='metric',
    order=SENTIMENT_LABELS,
)
plt.title('Final test precision, recall and F1 by class')
plt.xlabel('Sentiment')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.tight_layout()
class_metrics_path = OUTPUT_DIR / 'final_overall_final_test_class_metrics.png'
plt.savefig(class_metrics_path, dpi=160, bbox_inches='tight')
plt.show()

plt.figure(figsize=(7, 5.5))
sns.heatmap(cm_df, annot=True, fmt='d', cmap='Blues', cbar=False)
plt.title('Final test confusion matrix - counts')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.tight_layout()
cm_count_path = OUTPUT_DIR / 'final_overall_final_test_confusion_matrix_counts.png'
plt.savefig(cm_count_path, dpi=160, bbox_inches='tight')
plt.show()

plt.figure(figsize=(7, 5.5))
sns.heatmap(cm_norm_df, annot=True, fmt='.1%', cmap='Oranges', cbar=False, vmin=0, vmax=1)
plt.title('Final test confusion matrix - row normalized')
plt.xlabel('Predicted label')
plt.ylabel('True label')
plt.tight_layout()
cm_normalized_path = OUTPUT_DIR / 'final_overall_final_test_confusion_matrix_normalized.png'
plt.savefig(cm_normalized_path, dpi=160, bbox_inches='tight')
plt.show()

error_df = prediction_df[~prediction_df['is_correct']].copy()
error_pair_df = (
    error_df.groupby(['sentiment_label', 'pred_sentiment_label'])
    .size()
    .reset_index(name='count')
    .sort_values('count', ascending=False)
)
error_pair_df['pair'] = error_pair_df['sentiment_label'] + ' -> ' + error_pair_df['pred_sentiment_label']
display(error_pair_df)

plt.figure(figsize=(9, 4.8))
sns.barplot(data=error_pair_df, x='count', y='pair', color='#4c78a8')
plt.title('Most common final test confusion pairs')
plt.xlabel('Count')
plt.ylabel('True -> predicted')
plt.tight_layout()
error_pairs_path = OUTPUT_DIR / 'final_overall_final_test_error_pairs.png'
plt.savefig(error_pairs_path, dpi=160, bbox_inches='tight')
plt.show()

display(error_df[['review_id', 'sentiment_label', 'pred_sentiment_label', 'review']].head(20))

In [ ]:
final_model = Pipeline([
    ('features', final_features),
    ('clf', final_clf),
])

model_path = OUTPUT_DIR / 'tfidf_restaurant_review_3class_final_overall_model.joblib'
validation_results_path = OUTPUT_DIR / 'final_overall_validation_results.csv'
metrics_path = OUTPUT_DIR / 'final_overall_metrics.csv'
report_path = OUTPUT_DIR / 'final_overall_classification_report.csv'
class_metrics_table_path = OUTPUT_DIR / 'final_overall_class_metrics.csv'
cm_path = OUTPUT_DIR / 'final_overall_confusion_matrix.csv'
cm_norm_path = OUTPUT_DIR / 'final_overall_confusion_matrix_normalized.csv'
pred_path = OUTPUT_DIR / 'final_overall_predictions.csv'
error_pairs_csv_path = OUTPUT_DIR / 'final_overall_error_pairs.csv'
info_path = OUTPUT_DIR / 'final_overall_model_info.json'

joblib.dump(final_model, model_path)
validation_results_df.to_csv(validation_results_path, index=False)
metrics_df.to_csv(metrics_path, index=False)
report_df.to_csv(report_path)
class_metrics_df.to_csv(class_metrics_table_path, index=False)
cm_df.to_csv(cm_path)
cm_norm_df.to_csv(cm_norm_path)
prediction_df.to_csv(pred_path, index=False)
error_pair_df.to_csv(error_pairs_csv_path, index=False)

def to_jsonable(value):
    if isinstance(value, dict):
        return {str(k): to_jsonable(v) for k, v in value.items()}
    if isinstance(value, list):
        return [to_jsonable(v) for v in value]
    if isinstance(value, tuple):
        return [to_jsonable(v) for v in value]
    if isinstance(value, (np.integer,)):
        return int(value)
    if isinstance(value, (np.floating,)):
        return float(value)
    return value

model_info = {
    'dataset': 'vietnamese-restaurant-review-sentiment-dataset',
    'data_root': str(data_root),
    'task': '3-class Vietnamese restaurant review sentiment classification',
    'approach': 'Classical ML: TF-IDF word/char features + overall-metric model selection',
    'label_map': LABEL_MAP,
    'sentiment_labels_order': SENTIMENT_LABELS,
    'selection_score_weights': OVERALL_SCORE_WEIGHTS,
    'best_validation_config': best_config,
    'best_config_details': best_config_details,
    'final_class_weight': final_class_weight,
    'cleaning_stats': cleaning_stats,
    'final_test_metrics': final_test_metrics,
    'output_files': {
        'model': str(model_path),
        'validation_results': str(validation_results_path),
        'metrics': str(metrics_path),
        'classification_report': str(report_path),
        'class_metrics': str(class_metrics_table_path),
        'confusion_matrix': str(cm_path),
        'confusion_matrix_normalized': str(cm_norm_path),
        'predictions': str(pred_path),
        'error_pairs': str(error_pairs_csv_path),
        'class_distribution_chart': str(class_distribution_path),
        'validation_comparison_chart': str(validation_comparison_path),
        'class_metrics_chart': str(class_metrics_path),
        'confusion_matrix_counts_chart': str(cm_count_path),
        'confusion_matrix_normalized_chart': str(cm_normalized_path),
        'error_pairs_chart': str(error_pairs_path),
    },
}
info_path.write_text(json.dumps(to_jsonable(model_info), ensure_ascii=False, indent=2), encoding='utf-8')

print('Saved model:', model_path)
print('Saved validation results:', validation_results_path)
print('Saved metrics:', metrics_path)
print('Saved report:', report_path)
print('Saved confusion matrix:', cm_path)
print('Saved predictions:', pred_path)
print('Saved model info:', info_path)
print('Saved chart files in:', OUTPUT_DIR)

In [ ]:
def predict_sentiment(texts):
    texts_clean = [normalize_text(t) for t in texts]
    return final_model.predict(texts_clean)

sample_texts = [
    'Quán ăn rất ngon, nhân viên nhiệt tình, chắc chắn sẽ quay lại.',
    'Đồ ăn quá tệ, phục vụ lâu và thái độ không tốt.',
    'Món ăn tạm được nhưng giá hơi cao và không có gì đặc sắc.',
]

for text, pred in zip(sample_texts, predict_sentiment(sample_texts)):
    print(pred, '|', text)